1. Setup and Importing Packages

In [10]:
import pandas as pd 
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score

RANDOM_STATE = 1213

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_PATH = (PROJECT_ROOT / "data" / "processed" / "retailhero_causal_forest_features.csv")

df = pd.read_csv(PROCESSED_DATA_PATH)

print(df.shape)
display(df.head())

(200039, 25)


,client_id,age,gender_F,gender_M,gender_U,num_transactions,num_unique_products,num_stores_visited,total_quantity,avg_items_per_transaction,...,regular_points_spent,express_points_received,express_points_spent,net_regular_points_change,net_express_points_change,days_since_last_purchase,avg_days_between_transactions,customer_activity_span,treatment,target
0,000012768d,45,0,0,1,4,46,3,54.0,13.500000,...,0.0,0.0,0.0,25.7,0.0,4,34.441906,103,0,1
1,000036f903,72,1,0,0,32,96,5,169.0,5.281250,...,0.0,60.0,0.0,54.9,60.0,1,3.515704,108,1,1
2,00010925a5,83,0,0,1,18,58,2,79.0,4.388889,...,-17.0,0.0,0.0,14.8,0.0,10,6.049572,102,1,1
3,0001f552b0,33,1,0,0,15,79,4,106.0,7.066667,...,0.0,0.0,0.0,78.9,0.0,2,8.010879,112,1,1
4,00020e7b18,73,0,0,1,18,175,4,394.0,21.888889,...,-592.0,0.0,-30.0,-305.9,-30.0,3,6.597343,112,1,1


2. Definiting X, Y and D

In [11]:
D = df["treatment"] #marketing treatment
Y = df["target"] #purchase outcome

X = df.drop(columns=["client_id", "treatment", "target","gender_U"]) #customer characteristics pre-treatment

print("X:", X.shape)
print("D:", D.shape)
print("Y:", Y.shape)

# Sanity check: make sure that the treatment and outcome are binary
display(X.head())
print("Treatment distribution:")
display(D.value_counts(normalize=True).rename("proportion"))

print("Outcome distribution:")
display(Y.value_counts(normalize=True).rename("proportion"))

X: (200039, 21)
D: (200039,)
Y: (200039,)


,age,gender_F,gender_M,num_transactions,num_unique_products,num_stores_visited,total_quantity,avg_items_per_transaction,total_spend,avg_transaction_spend,...,spend_std,regular_points_received,regular_points_spent,express_points_received,express_points_spent,net_regular_points_change,net_express_points_change,days_since_last_purchase,avg_days_between_transactions,customer_activity_span
0,45,0,0,4,46,3,54.0,13.500000,2805.0,701.250000,...,257.969475,25.7,0.0,0.0,0.0,25.7,0.0,4,34.441906,103
1,72,1,0,32,96,5,169.0,5.281250,9810.0,306.562500,...,161.829248,54.9,0.0,60.0,0.0,54.9,60.0,1,3.515704,108
2,83,0,0,18,58,2,79.0,4.388889,5873.0,326.277778,...,138.784558,31.8,-17.0,0.0,0.0,14.8,0.0,10,6.049572,102
3,33,1,0,15,79,4,106.0,7.066667,6155.0,410.333333,...,295.879721,78.9,0.0,0.0,0.0,78.9,0.0,2,8.010879,112
4,73,0,0,18,175,4,394.0,21.888889,25206.0,1400.333333,...,1034.475201,286.1,-592.0,0.0,-30.0,-305.9,-30.0,3,6.597343,112


Treatment distribution:


treatment
0    0.500192
1    0.499808
Name: proportion, dtype: float64

Outcome distribution:


target
1    0.619889
0    0.380111
Name: proportion, dtype: float64

3. 80/20 train-test split

In [12]:
strata = D.astype(str) + "_" + Y.astype(str)
(x_train, x_test, d_train, d_test, y_train, y_test) = train_test_split(X,D,Y, test_size=0.2, random_state=RANDOM_STATE, stratify=strata)

# sanity check, stratification works 
print(f"Training observations: {len(x_train):,}")
print(f"Test observations:     {len(x_test):,}")

print("\nTreatment rates:")
print(f"Full:  {D.mean():.4f}")
print(f"Train: {d_train.mean():.4f}")
print(f"Test:  {d_test.mean():.4f}")

print("\nPurchase rates:")
print(f"Full:  {Y.mean():.4f}")
print(f"Train: {y_train.mean():.4f}")
print(f"Test:  {y_test.mean():.4f}")



Training observations: 160,031
Test observations:     40,008

Treatment rates:
Full:  0.4998
Train: 0.4998
Test:  0.4998

Purchase rates:
Full:  0.6199
Train: 0.6199
Test:  0.6199


4. Standardisation

In [13]:
feature_info = pd.DataFrame({
    "dtype": X.dtypes,
    "n_unique": X.nunique(),
    "min": X.min(),
    "max": X.max()
})

#display(feature_info)

binary_cols = [
    col for col in X.columns
    if set(X[col].dropna().unique()).issubset({0, 1})
]

# identify continuous and binary columns

continuous_cols = [
    col for col in X.columns
    if col not in binary_cols
]

print("Binary variables:")
print(binary_cols)

print("\nContinuous/count variables:")
print(continuous_cols)

# stadardise continuous variables

scaler = StandardScaler()

x_train_scaled = x_train.copy()
x_test_scaled = x_test.copy()

x_train_scaled[continuous_cols] = scaler.fit_transform(x_train[continuous_cols])
x_test_scaled[continuous_cols] = scaler.transform(x_test[continuous_cols])

# sanity check for normalisation

display(x_train_scaled[continuous_cols].describe().loc[["mean", "std"]])

Binary variables:
['gender_F', 'gender_M']

Continuous/count variables:
['age', 'num_transactions', 'num_unique_products', 'num_stores_visited', 'total_quantity', 'avg_items_per_transaction', 'total_spend', 'avg_transaction_spend', 'median_transaction_spend', 'spend_std', 'regular_points_received', 'regular_points_spent', 'express_points_received', 'express_points_spent', 'net_regular_points_change', 'net_express_points_change', 'days_since_last_purchase', 'avg_days_between_transactions', 'customer_activity_span']


,age,num_transactions,num_unique_products,num_stores_visited,total_quantity,avg_items_per_transaction,total_spend,avg_transaction_spend,median_transaction_spend,spend_std,regular_points_received,regular_points_spent,express_points_received,express_points_spent,net_regular_points_change,net_express_points_change,days_since_last_purchase,avg_days_between_transactions,customer_activity_span
mean,4.178070e-17,9.084305e-17,4.297951e-17,-9.315187e-17,-4.653153e-17,4.011125e-16,-9.856871e-18,4.932875e-17,5.268098e-17,5.043876e-17,1.443898e-16,5.927443e-18,1.758253e-17,-2.592979e-17,1.296489e-17,2.308817e-17,-8.325060e-18,-9.839111e-17,2.415377e-17
std,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00


5. Construction of treatment interactions


In [14]:
d_train = d_train.loc[x_train_scaled.index]
d_test = d_test.loc[x_test_scaled.index]

x_train_interactions = x_train_scaled.mul(d_train, axis=0) 
x_test_interactions = x_test_scaled.mul(d_test, axis=0)

x_train_interactions.columns = [f"{col}_treatment" for col in x_train_interactions.columns]
x_test_interactions.columns = [f"{col}_treatment" for col in x_test_interactions.columns]

display(x_train_interactions.head())

d_train_df = d_train.rename("treatment").to_frame()
d_test_df = d_test.rename("treatment").to_frame()

z_train = pd.concat([d_train_df, x_train_scaled, x_train_interactions], axis=1)

z_test = pd.concat([d_test_df, x_test_scaled, x_test_interactions], axis=1)

print("training design matrix shape:", z_train.shape)
print("test design matrix shape:", z_test.shape)

display(z_train.head())

,age_treatment,gender_F_treatment,gender_M_treatment,num_transactions_treatment,num_unique_products_treatment,num_stores_visited_treatment,total_quantity_treatment,avg_items_per_transaction_treatment,total_spend_treatment,avg_transaction_spend_treatment,...,spend_std_treatment,regular_points_received_treatment,regular_points_spent_treatment,express_points_received_treatment,express_points_spent_treatment,net_regular_points_change_treatment,net_express_points_change_treatment,days_since_last_purchase_treatment,avg_days_between_transactions_treatment,customer_activity_span_treatment
20830,-0.000000,0,0,-0.000000,-0.000000,0.000000,-0.000000,-0.000000,-0.000000,0.000000,...,0.000000,-0.000000,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
30774,-0.714523,1,0,-0.062622,0.880188,1.037885,0.130507,0.082943,0.140806,0.055197,...,-0.159947,0.139179,-0.616720,-0.106004,0.394456,-0.757566,0.328179,0.736336,-0.319194,0.460040
7400,0.605930,0,1,1.628556,-0.040488,0.539057,-0.004352,-0.939519,1.478552,-0.145452,...,-0.270384,1.423589,-1.547371,-0.106004,0.394456,-0.709460,0.328179,-0.321910,-0.656527,0.635997
8004,-0.000000,0,0,-0.000000,-0.000000,0.000000,-0.000000,0.000000,-0.000000,-0.000000,...,-0.000000,-0.000000,0.000000,-0.000000,0.000000,-0.000000,0.000000,-0.000000,0.000000,0.000000
63446,-0.085735,0,0,-0.288113,1.517580,-0.957427,2.068219,3.834782,1.419772,2.434741,...,1.607046,2.270109,-2.984961,-0.106004,0.394456,-1.896464,0.328179,-0.674659,-0.228615,0.108126


training design matrix shape: (160031, 43)
test design matrix shape: (40008, 43)


,treatment,age,gender_F,gender_M,num_transactions,num_unique_products,num_stores_visited,total_quantity,avg_items_per_transaction,total_spend,...,spend_std_treatment,regular_points_received_treatment,regular_points_spent_treatment,express_points_received_treatment,express_points_spent_treatment,net_regular_points_change_treatment,net_express_points_change_treatment,days_since_last_purchase_treatment,avg_days_between_transactions_treatment,customer_activity_span_treatment
20830,0,-1.091795,0,0,-0.569976,-0.589353,0.539057,-0.515396,-0.187297,-0.346528,...,0.000000,-0.000000,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
30774,1,-0.714523,1,0,-0.062622,0.880188,1.037885,0.130507,0.082943,0.140806,...,-0.159947,0.139179,-0.616720,-0.106004,0.394456,-0.757566,0.328179,0.736336,-0.319194,0.460040
7400,1,0.605930,0,1,1.628556,-0.040488,0.539057,-0.004352,-0.939519,1.478552,...,-0.270384,1.423589,-1.547371,-0.106004,0.394456,-0.709460,0.328179,-0.321910,-0.656527,0.635997
8004,0,-0.777401,1,0,-0.626348,-0.660174,0.539057,-0.494103,0.016456,-0.614793,...,-0.000000,-0.000000,0.000000,-0.000000,0.000000,-0.000000,0.000000,-0.000000,0.000000,0.000000
63446,1,-0.085735,0,0,-0.288113,1.517580,-0.957427,2.068219,3.834782,1.419772,...,1.607046,2.270109,-2.984961,-0.106004,0.394456,-1.896464,0.328179,-0.674659,-0.228615,0.108126


6. Cross Validation + Lasso

In [15]:
# 5-fold cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Candidate C values
# Smaller C = stronger regularisation
c_grid = np.logspace(-3, 2, 20)

# Fit cross-validated LASSO logistic regression
lasso_cv = LogisticRegressionCV(
    Cs=c_grid,
    cv=cv,
    penalty="l1",
    solver="saga",
    scoring="neg_log_loss",
    random_state=RANDOM_STATE,
    refit=True,
    max_iter=5000,
    n_jobs=-1
)

lasso_cv.fit(z_train, y_train)

# ---------------------------------------------------------
# Apply the 1-standard-error (1-SE) rule
# ---------------------------------------------------------

score_key = list(lasso_cv.scores_.keys())[0]
cv_scores = lasso_cv.scores_[score_key]

# Mean CV score and standard error for each C
mean_scores = cv_scores.mean(axis=0)
se_scores = cv_scores.std(axis=0, ddof=1) / np.sqrt(cv_scores.shape[0])

# Best-performing C
best_idx = np.argmax(mean_scores)
best_c = lasso_cv.Cs_[best_idx]
best_score = mean_scores[best_idx]
best_se = se_scores[best_idx]

# 1-SE threshold
one_se_threshold = best_score - best_se

# Find all C values whose performance is within 1 SE of the best
eligible_idx = np.where(mean_scores >= one_se_threshold)[0]

# Choose smallest C = strongest regularisation within 1 SE
one_se_c = lasso_cv.Cs_[eligible_idx[0]]

print(f"CV-optimal C: {best_c:.6f}")
print(f"1-SE C:       {one_se_c:.6f}")

# ---------------------------------------------------------
# Refit LASSO using the more parsimonious 1-SE C
# ---------------------------------------------------------

lasso_1se = LogisticRegression(
    penalty="l1",
    solver="saga",
    C=one_se_c,
    random_state=RANDOM_STATE,
    max_iter=5000
)

lasso_1se.fit(z_train, y_train)

CV-optimal C: 0.127427
1-SE C:       0.003360


,penalty,'l1'
,dual,False
,tol,0.0001
,C,np.float64(0....9818286283781)
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,1213
,solver,'saga'
,max_iter,5000
,multi_class,'deprecated'


In [16]:
# Extract coefficients from the 1-SE LASSO model
coef_df = pd.DataFrame({
    "variable": z_train.columns,
    "coefficient": lasso_1se.coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df = coef_df.sort_values(
    by="abs_coefficient",
    ascending=False
)

# Keep only treatment interaction terms
interaction_results = coef_df[
    coef_df["variable"].str.endswith("_treatment")
].copy()

# LASSO-selected interactions have non-zero coefficients
interaction_results["selected"] = (
    interaction_results["abs_coefficient"] > 1e-8
)

interaction_results = interaction_results.sort_values(
    by="abs_coefficient",
    ascending=False
)

print("All treatment interactions:")
display(interaction_results)

# Keep only selected interactions
selected_interactions = interaction_results[
    interaction_results["selected"]
].copy()

print(
    f"\n{len(selected_interactions)} out of "
    f"{len(interaction_results)} treatment interactions selected "
    f"using the 1-SE rule."
)

print("\nSelected treatment interactions:")
display(
    selected_interactions[
        ["variable", "coefficient", "abs_coefficient"]
    ]
)

All treatment interactions:


,variable,coefficient,abs_coefficient,selected
37,express_points_spent_treatment,-0.047088,0.047088,True
22,age_treatment,0.036510,0.036510,True
23,gender_F_treatment,0.017998,0.017998,True
33,spend_std_treatment,-0.010940,0.010940,True
39,net_express_points_change_treatment,-0.005956,0.005956,True
27,num_stores_visited_treatment,-0.003550,0.003550,True
26,num_unique_products_treatment,0.000000,0.000000,False
25,num_transactions_treatment,0.000000,0.000000,False
24,gender_M_treatment,0.000000,0.000000,False
30,total_spend_treatment,0.000000,0.000000,False



6 out of 21 treatment interactions selected using the 1-SE rule.

Selected treatment interactions:


,variable,coefficient,abs_coefficient
37,express_points_spent_treatment,-0.047088,0.047088
22,age_treatment,0.036510,0.036510
23,gender_F_treatment,0.017998,0.017998
33,spend_std_treatment,-0.010940,0.010940
39,net_express_points_change_treatment,-0.005956,0.005956
27,num_stores_visited_treatment,-0.003550,0.003550


7. Main takeaways

In [17]:
selected_features = (selected_interactions["variable"].str.replace("_treatment", "", regex=False).tolist()
)

main_effects = X.columns.tolist()

print("=" * 60)
print("LASSO INTERACTION SELECTION: CONCLUSION")
print("=" * 60)

print(f"\nLASSO selected {len(selected_features)} out of "f"{len(interaction_results)} candidate treatment interactions ""using 5-fold cross-validation and the 1-SE rule.")

print("\nSelected treatment interactions:")
for feature in selected_features:
    print(f"  - {feature} x treatment")

print("\nMain effects retained:")
for feature in main_effects:
    print(f"  - {feature}")

print(
    f"\nFinal LASSO-informed logistic specification:"
    f"\n  - 1 treatment indicator"
    f"\n  - {len(main_effects)} customer main effects"
    f"\n  - {len(selected_features)} selected treatment interactions"
    f"\n  - {1 + len(main_effects) + len(selected_features)} predictors in total"
)

print(
    "\nNext step: Fit a logistic regression using the treatment indicator, "
    "all customer main effects, and the LASSO-selected treatment interactions. "
    "The resulting heterogeneity model can then be compared with the "
    "pre-specified logistic model and causal forest."
)

LASSO INTERACTION SELECTION: CONCLUSION

LASSO selected 6 out of 21 candidate treatment interactions using 5-fold cross-validation and the 1-SE rule.

Selected treatment interactions:
  - express_points_spent x treatment
  - age x treatment
  - gender_F x treatment
  - spend_std x treatment
  - net_express_points_change x treatment
  - num_stores_visited x treatment

Main effects retained:
  - age
  - gender_F
  - gender_M
  - num_transactions
  - num_unique_products
  - num_stores_visited
  - total_quantity
  - avg_items_per_transaction
  - total_spend
  - avg_transaction_spend
  - median_transaction_spend
  - spend_std
  - regular_points_received
  - regular_points_spent
  - express_points_received
  - express_points_spent
  - net_regular_points_change
  - net_express_points_change
  - days_since_last_purchase
  - avg_days_between_transactions
  - customer_activity_span

Final LASSO-informed logistic specification:
  - 1 treatment indicator
  - 21 customer main effects
  - 6 selected